# Computing 3D Point Spread Functions

This notebook demonstrates the computation of 3D PSFs along the optical axis and the use of vectorial propagators for high numerical aperture systems.

## Import Libraries

In [ ]:
import math
from psf_generator.propagators import ScalarSphericalPropagator, VectorialSphericalPropagator
from psf_generator.utils.plots import plot_psf

## Example 1: Computing a 3D Scalar PSF

To compute the field at multiple positions along the optical axis, we specify:
- `defocus_step`: Axial spacing between z-slices (nm)
- `n_defocus`: Number of z-slices

The total z-range is $(n_{\mathrm{defocus}} - 1) \times \Delta z$, centered at the focal plane.

In [ ]:
base_kwargs_3d = {
    'n_pix_pupil': 127,
    'n_pix_psf': 256,
    'na': 1.3,
    'wavelength': 480,
    'pix_size': 10,
    'defocus_step': 25,
    'n_defocus': 301,
}

### Computation and Visualization

The visualization shows three orthogonal cross-sections through the 3D PSF: xy (focal plane), xz, and yz.

In [ ]:
propagator = ScalarSphericalPropagator(**base_kwargs_3d)
psf_3d = propagator.compute_focus_field()

for quantity in ['modulus', 'phase', 'intensity']:
    plot_psf(
        psf=psf_3d,
        name_of_propagator=propagator.get_name(),
        quantity=quantity,
        show_titles=True,
        show_cbar_ticks=True
    )

The xz and yz cross-sections reveal the characteristic axial elongation of the PSF, with axial resolution typically 2-3× worse than lateral resolution.

## Example 2: Vectorial PSF for High NA Systems

At high numerical apertures (NA > 0.9), the scalar wave approximation breaks down. The vectorial nature of the electromagnetic field must be considered, with field components (Ex, Ey, Ez) computed using the `VectorialSphericalPropagator`.

### Polarization State

The input polarization is specified by complex amplitudes `e0x` and `e0y`. Here we use circular polarization:
$$(e_0^x, e_0^y) = \left(\frac{\sqrt{2}}{2}, \frac{\sqrt{2}}{2}i\right)$$

In [ ]:
vectorial_kwargs = {
    **base_kwargs_3d,
    'e0x': math.sqrt(2) / 2,
    'e0y': math.sqrt(2) / 2 * 1j,
}

propagator_vec = VectorialSphericalPropagator(**vectorial_kwargs)
psf_vec = propagator_vec.compute_focus_field()

for quantity in ['modulus', 'phase', 'intensity']:
    plot_psf(
        psf=psf_vec,
        name_of_propagator=propagator_vec.get_name(),
        quantity=quantity,
        show_titles=True,
        show_cbar_ticks=True
    )

The vectorial output contains three field components. For modulus and phase, each component (Ex, Ey, Ez) is shown separately. The intensity is computed as $I = |E_x|^2 + |E_y|^2 + |E_z|^2$.

## Troubleshooting

**Performance optimization:**
- Reduce `n_defocus` or increase `defocus_step` to decrease computational cost
- Reduce `n_pix_pupil` for faster computation
- Add `'device': 'cuda:0'` for GPU acceleration
- Note: Vectorial propagators are ~3× slower than scalar

**Memory management:**
- Array size: `(n_defocus, n_pix_psf, n_pix_psf)` for scalar
- Array size: `(3, n_defocus, n_pix_psf, n_pix_psf)` for vectorial
- Reduce `n_defocus` and `n_pix_psf` to manage memory footprint

**Z-range specification:**
- Total range: $(n_{\mathrm{defocus}} - 1) \times \Delta z$
- For ±3 μm: use `n_defocus=241, defocus_step=25` nm

**Accuracy considerations:**
- Increase `n_pix_pupil` (127, 255, or 511) for numerical convergence
- Ensure adequate PSF sampling: typically `pix_size` = 5-20 nm

**Visualizing specific z-slices:**
- Use `plot_psf(..., z_slice_number=n)` to display a specific plane
- Center slice (focal plane): `z_slice_number = n_defocus // 2`

**Propagator selection:**
- `ScalarSphericalPropagator`: NA < 0.9
- `VectorialSphericalPropagator`: NA ≥ 0.9, accounts for polarization
- `ScalarCartesianPropagator`: Arbitrary phase masks, NA < 0.9
- `VectorialCartesianPropagator`: Non-axisymmetric aberrations, NA ≥ 0.9, accounts for polarization